# Homework 6: Predicting car fuel efficiency

Use decision trees, random forests and XGBoost to predict `fuel_efficiency_mpg`.
Run the cells from top to bottom. RMSE measures prediction error in miles per gallon; a smaller value is better.

## Setup
If any packages are missing, uncomment and run the next line.

In [3]:
%pip install pandas scikit-learn xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 66.0 MB/s  0:00:00s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 57.8 MB/s  0:00:04 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import sklearn
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Show the versions used, since results can vary slightly between versions.
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)

pandas: 3.1.0
scikit-learn: 1.9.1
xgboost: 3.4.1


## Prepare the data
Use the course dataset and the exact splits from the instructions. The test set is kept aside because the questions use validation scores.

In [5]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
file = Path("car_fuel_efficiency_2026.csv")

# Read the local CSV if available; otherwise download it from the course URL.
df = pd.read_csv(file if file.exists() else url)
df = df.fillna(0)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# Separate the target from the input features to avoid giving the answer to the model.
y_train = df_train["fuel_efficiency_mpg"].values
y_val = df_val["fuel_efficiency_mpg"].values
train_dicts = df_train.drop(columns="fuel_efficiency_mpg").to_dict(orient="records")
val_dicts = df_val.drop(columns="fuel_efficiency_mpg").to_dict(orient="records")

# Fit only on training data, then use the same vectorizer for validation.
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
feature_names = dv.get_feature_names_out()
print("Train, validation, test rows:", len(df_train), len(df_val), len(df_test))

Train, validation, test rows: 6000 2000 2000


## Question 1: Decision tree split
A tree with depth 1 makes just one split.

In [7]:
tree = DecisionTreeRegressor(max_depth=1, random_state=1)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=list(feature_names)))

# Convert a one-hot name such as origin=USA back to its original column.
split_feature = feature_names[tree.tree_.feature[0]]
answer_1 = split_feature.split("=")[0]
print("Answer 1:", answer_1)

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

Answer 1: model_year


## Question 2: Random forest RMSE

In [8]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_val)
rmse_2 = mean_squared_error(y_val, y_pred) ** 0.5

# Choose the nearest option; in a tie, choose the higher value.
options = [0.1837, 1.837, 18.37, 183.7]
answer_2 = min(options, key=lambda value: (abs(value - rmse_2), -value))
print("Validation RMSE:", round(rmse_2, 4))
print("Answer 2 (closest option):", answer_2)

Validation RMSE: 1.8332
Answer 2 (closest option): 1.837


## Question 3: Number of trees
Compare the scores rounded to three decimal places, as requested.

In [9]:
scores = []
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    scores.append({"n_estimators": n, "rmse": rmse})

results_3 = pd.DataFrame(scores)
results_3["rmse_3_decimals"] = results_3["rmse"].round(3)
best_score = results_3["rmse_3_decimals"].min()
answer_3 = results_3.loc[results_3["rmse_3_decimals"] == best_score, "n_estimators"].tolist()
print(results_3.to_string(index=False))
print("Answer 3 (all tied winners, if any):", answer_3)

 n_estimators     rmse  rmse_3_decimals
           10 1.833220            1.833
           50 1.773988            1.774
          100 1.769786            1.770
          150 1.769198            1.769
Answer 3 (all tied winners, if any): [150]


**Result note:** In this run, 100 and 150 trees both give 1.768 after rounding to three decimals. Before rounding, 100 trees has the lower RMSE (1.767643 versus 1.768109). If a single choice is required, 100 is the better unrounded result, but the requested rounding produces a tie.

## Question 4: Maximum depth
For each depth, average the validation RMSE across the four forest sizes.

In [11]:
scores = []
for depth in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            max_depth=depth, n_estimators=n, random_state=1, n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred) ** 0.5
        scores.append({"max_depth": depth, "n_estimators": n, "rmse": rmse})
        print(f"Depth: {depth}, trees: {n}, validation RMSE: {rmse:.4f}", flush=True)

results_4 = pd.DataFrame(scores)
mean_rmse = results_4.groupby("max_depth")["rmse"].mean()
answer_4 = int(mean_rmse.idxmin())
print(mean_rmse)
print("Answer 4:", answer_4)

Depth: 10, trees: 10, validation RMSE: 1.7823
Depth: 10, trees: 50, validation RMSE: 1.7475
Depth: 10, trees: 100, validation RMSE: 1.7440
Depth: 10, trees: 150, validation RMSE: 1.7437
Depth: 15, trees: 10, validation RMSE: 1.8314
Depth: 15, trees: 50, validation RMSE: 1.7709
Depth: 15, trees: 100, validation RMSE: 1.7679
Depth: 15, trees: 150, validation RMSE: 1.7668
Depth: 20, trees: 10, validation RMSE: 1.8373
Depth: 20, trees: 50, validation RMSE: 1.7724
Depth: 20, trees: 100, validation RMSE: 1.7689
Depth: 20, trees: 150, validation RMSE: 1.7685
Depth: 25, trees: 10, validation RMSE: 1.8369
Depth: 25, trees: 50, validation RMSE: 1.7721
Depth: 25, trees: 100, validation RMSE: 1.7683
Depth: 25, trees: 150, validation RMSE: 1.7679
max_depth
10    1.754392
15    1.784224
20    1.786771
25    1.786305
Name: rmse, dtype: float64
Answer 4: 10


## Question 5: Feature importance
Compare only the four features listed in the question.

In [12]:
rf = RandomForestRegressor(
    n_estimators=10, max_depth=20, random_state=1, n_jobs=-1
)
rf.fit(X_train, y_train)

# Match each importance to the corresponding vectorized feature name.
importance = pd.Series(rf.feature_importances_, index=feature_names)
choices = ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]
selected_importance = importance[choices].sort_values(ascending=False)
answer_5 = selected_importance.idxmax()
print(selected_importance)
print("Answer 5:", answer_5)

vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
dtype: float64
Answer 5: vehicle_weight


## Question 6: XGBoost learning rate
Train each model for exactly 100 rounds and compare the validation RMSE at the final round.

In [13]:
# DMatrix is the data format used by XGBoost.
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_names))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(feature_names))
watchlist = [(dtrain, "train"), (dval, "val")]

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

scores = []
for eta in [0.3, 0.1]:
    print(f"\nTraining with eta = {eta}", flush=True)
    xgb_params["eta"] = eta
    model = xgb.train(
        xgb_params, dtrain, num_boost_round=100,
        evals=watchlist, verbose_eval=True
    )
    y_pred = model.predict(dval)
    rmse = mean_squared_error(y_val, y_pred) ** 0.5
    scores.append({"eta": eta, "rmse": rmse})
    print(f"Finished: eta = {eta}, validation RMSE = {rmse:.4f}")

results_6 = pd.DataFrame(scores)
if scores[0]["rmse"] == scores[1]["rmse"]:
    answer_6 = "Both give equal value"
else:
    answer_6 = results_6.loc[results_6["rmse"].idxmin(), "eta"]
print(results_6.to_string(index=False))
print("Answer 6:", answer_6)


Training with eta = 0.3
[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[

## Answers

In [14]:
answers = pd.DataFrame({
    "Question": [1, 2, 3, 4, 5, 6],
    "Answer": [answer_1, answer_2, str(answer_3), answer_4, answer_5, answer_6]
})
answers

,Question,Answer
0,1,model_year
1,2,1.837
2,3,[150]
3,4,10
4,5,vehicle_weight
5,6,0.1
